In [1]:
import os
import random
import numpy as np

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
train_path = "MNIST/train"

classes = os.listdir(train_path)

print(classes)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [3]:
class CustomDataset:

    def __init__(self, root):

        self.samples = []

        classes = sorted(os.listdir(root))

        for label in classes:

            folder = os.path.join(root, label)

            for image_name in os.listdir(folder):

                image_path = os.path.join(folder, image_name)

                self.samples.append(
                    (image_path, int(label))
                )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        image = Image.open(image_path).convert("L")

        image = np.array(image, dtype=np.float32)/255.0

        image = torch.tensor(image).view(-1)

        label = torch.tensor(label)

        return image, label

In [4]:
train_dataset = CustomDataset("MNIST/train")

print(len(train_dataset))

60000


In [5]:
class CustomDataLoader:

    def __init__(self, dataset, batch_size=32, shuffle=True):

        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle

        self.indices = list(range(len(dataset)))
    def __len__(self):

        return (len(self.dataset) + self.batch_size - 1) // self.batch_size
    def __iter__(self):

        if self.shuffle:
            random.shuffle(self.indices)

        self.current = 0

        return self

    def __next__(self):

        if self.current >= len(self.indices):
            raise StopIteration

        batch_indices = self.indices[
            self.current : self.current + self.batch_size
        ]

        images = []
        labels = []

        for idx in batch_indices:

            image, label = self.dataset[idx]

            images.append(image)
            labels.append(label)

        self.current += self.batch_size

        images = torch.stack(images)

        labels = torch.tensor(labels)

        return images, labels

In [6]:
trainloader = CustomDataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [7]:
images, labels = next(iter(trainloader))

print(images.shape)

print(labels.shape)

torch.Size([64, 784])
torch.Size([64])


In [8]:
for images, labels in trainloader:

    print(images.shape)

    print(labels.shape)

    break

torch.Size([64, 784])
torch.Size([64])


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)


cuda


In [10]:
class MLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(784,256)

        self.fc2 = nn.Linear(256,128)

        self.fc3 = nn.Linear(128,10)

        self.relu = nn.ReLU()

    def forward(self,x):

        x = self.relu(self.fc1(x))

        x = self.relu(self.fc2(x))

        x = self.fc3(x)

        return x

In [11]:
model = MLP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [12]:
test_dataset = CustomDataset("MNIST/test")

testloader = CustomDataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [13]:
def train(model, trainloader, testloader, epochs):

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for images, labels in trainloader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        # -------------------------
        # Evaluation
        # -------------------------

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in testloader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, pred = torch.max(outputs,1)

                total += labels.size(0)

                correct += (pred==labels).sum().item()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss={running_loss/len(trainloader):.4f} | "
            f"Accuracy={100*correct/total:.2f}%"
        )

In [14]:
train(
    model,
    trainloader,
    testloader,
    epochs=5
)

Epoch 1/5 | Loss=0.2811 | Accuracy=95.82%
Epoch 2/5 | Loss=0.1052 | Accuracy=97.21%
Epoch 3/5 | Loss=0.0717 | Accuracy=97.50%
Epoch 4/5 | Loss=0.0501 | Accuracy=97.68%
Epoch 5/5 | Loss=0.0400 | Accuracy=97.66%
